# 🔋 Battery SoH Estimation — Week 3 (Updated)

**WIDSS — Windowed Intelligent Drive-cycle State eStimation**

This notebook extends Week 3 with:
1. Multi-configuration battery robustness testing
2. Hyperparameter tuning (GridSearchCV)
3. Stacked / deeper LSTM with dropout regularisation
4. Error analysis and residual diagnostics
5. Coulomb-counting drift comparison


## 0. Setup

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from widss.simulation import BatterySimulationConfig, build_dataset
from widss.dataset import build_sequences
from widss.evaluation import rmse, mae, mape

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})
SEED = 42
print('✅ Ready')

---
## 1. Multi-Configuration Robustness Testing

Train on one battery config, test on unseen configurations to measure generalisation.

In [ ]:
# Define a range of battery configurations
CONFIGS = {
    'Standard 60Ah': BatterySimulationConfig(
        capacity_ah=60.0, soc_init=0.95, internal_resistance_ohm=0.02),
    'High Cap 80Ah' : BatterySimulationConfig(
        capacity_ah=80.0, soc_init=0.90, internal_resistance_ohm=0.015),
    'Aged 50Ah'     : BatterySimulationConfig(
        capacity_ah=50.0, soc_init=0.80, internal_resistance_ohm=0.05),
    'Small 40Ah'    : BatterySimulationConfig(
        capacity_ah=40.0, soc_init=0.70, internal_resistance_ohm=0.03),
}

datasets = {}
for name, cfg in CONFIGS.items():
    df = build_dataset(duration_s=3600, config=cfg, seed=SEED)
    datasets[name] = df
    print(f'{name:20s} → {len(df)} rows | SoC: [{df.soc.min():.3f}, {df.soc.max():.3f}]')

In [ ]:
# ── Overlay SoC trajectories across configs ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['steelblue', 'darkorange', 'green', 'crimson']
for (name, df), color in zip(datasets.items(), colors):
    t = df['time_s'] / 60  # minutes
    axes[0].plot(t, df['soc'], color=color, lw=1.2, label=name)
    axes[1].plot(t, df['voltage_v'], color=color, lw=1.2, label=name)

axes[0].set_title('SoC Trajectories (different configs)')
axes[0].set_xlabel('Time (min)'); axes[0].set_ylabel('SoC')
axes[0].legend(fontsize=9)

axes[1].set_title('Voltage Profiles')
axes[1].set_xlabel('Time (min)'); axes[1].set_ylabel('Voltage (V)')
axes[1].legend(fontsize=9)

plt.tight_layout(); plt.show()

---
## 2. Feature Engineering Helper

In [ ]:
def engineer_features(df: pd.DataFrame, dt_s: float = 1.0, win: int = 60) -> pd.DataFrame:
    """Add rolling stats, cumulative charge, power, and lag features."""
    d = df.copy()
    d['current_roll_mean'] = d['current_a'].rolling(win, min_periods=1).mean()
    d['current_roll_std']  = d['current_a'].rolling(win, min_periods=1).std().fillna(0)
    d['voltage_roll_mean'] = d['voltage_v'].rolling(win, min_periods=1).mean()
    d['charge_ah']  = (d['current_a'] * dt_s / 3600).cumsum()
    d['power_w']    = d['current_a'] * d['voltage_v']
    for lag in [1, 5, 30]:
        d[f'current_lag{lag}'] = d['current_a'].shift(lag).fillna(0)
        d[f'voltage_lag{lag}'] = d['voltage_v'].shift(lag).fillna(method='bfill')
    return d

FEATURE_COLS = [
    'current_a', 'voltage_v',
    'current_roll_mean', 'current_roll_std', 'voltage_roll_mean',
    'charge_ah', 'power_w',
    'current_lag1', 'voltage_lag1',
    'current_lag5', 'voltage_lag5',
    'current_lag30', 'voltage_lag30'
]
TARGET = 'soc'

# Apply to all datasets
feat_datasets = {name: engineer_features(df) for name, df in datasets.items()}
print('Feature engineering complete')

---
## 3. Hyperparameter Tuning — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

# Use Standard 60Ah config for tuning
df_train_cfg = feat_datasets['Standard 60Ah']
split = int(0.8 * len(df_train_cfg))

X_all = df_train_cfg[FEATURE_COLS].values
y_all = df_train_cfg[TARGET].values

X_tr, y_tr = X_all[:split], y_all[:split]
X_te, y_te = X_all[split:], y_all[split:]

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc  = scaler.transform(X_te)

# TimeSeriesSplit respects temporal order — no data leakage
tscv = TimeSeriesSplit(n_splits=3)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth'   : [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf = RandomForestRegressor(random_state=SEED, n_jobs=-1)
gs = GridSearchCV(rf, param_grid, cv=tscv, scoring='neg_root_mean_squared_error',
                  verbose=1, refit=True)
gs.fit(X_tr_sc, y_tr)

print(f'\nBest params : {gs.best_params_}')
print(f'Best CV RMSE: {-gs.best_score_:.4f}')

In [ ]:
# Evaluate best RF on test set
best_rf = gs.best_estimator_
y_rf_pred = np.clip(best_rf.predict(X_te_sc), 0, 1)

print('── Tuned Random Forest (Test) ────────────────')
print(f'RMSE : {rmse(y_te, y_rf_pred):.4f}')
print(f'MAE  : {mae(y_te, y_rf_pred):.4f}')
print(f'MAPE : {mape(y_te, y_rf_pred):.2f} %')

In [ ]:
# GridSearch RMSE heatmap (n_estimators × max_depth for min_samples_split=2)
cv_results = pd.DataFrame(gs.cv_results_)
pivot_data = cv_results[
    cv_results['param_min_samples_split'] == 2
].pivot_table(
    index='param_max_depth',
    columns='param_n_estimators',
    values='mean_test_score'
) * -1  # convert neg_rmse → rmse

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot_data, annot=True, fmt='.4f', cmap='YlOrRd_r', ax=ax)
ax.set_title('CV RMSE — n_estimators vs max_depth (min_samples_split=2)')
plt.tight_layout(); plt.show()

---
## 4. Cross-Config Generalisation Test

In [ ]:
# Train on Standard 60Ah, test on all other configs
gen_results = {}

for cfg_name, df_cfg in feat_datasets.items():
    X_cfg = scaler.transform(df_cfg[FEATURE_COLS].values)
    y_cfg = df_cfg[TARGET].values
    y_pred_cfg = np.clip(best_rf.predict(X_cfg), 0, 1)
    gen_results[cfg_name] = {
        'RMSE': rmse(y_cfg, y_pred_cfg),
        'MAE' : mae(y_cfg, y_pred_cfg),
        'MAPE': mape(y_cfg, y_pred_cfg)
    }

gen_df = pd.DataFrame(gen_results).T.round(4)
print('Generalisation (trained on Standard 60Ah):')
gen_df

In [ ]:
# Bar chart of cross-config RMSE
fig, ax = plt.subplots(figsize=(8, 4))
gen_df['RMSE'].plot(kind='bar', ax=ax, color=['steelblue','darkorange','green','crimson'],
                   edgecolor='white')
ax.set_title('Cross-Config RMSE (Random Forest)')
ax.set_ylabel('RMSE')
ax.set_xlabel('Battery Configuration')
ax.tick_params(axis='x', rotation=15)
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()

---
## 5. Stacked LSTM with Dropout

In [ ]:
from widss.model import tensorflow_available

WINDOW_SIZE = 60  # wider context window than Week 3 baseline

if not tensorflow_available():
    print('⚠️  TensorFlow not installed — skipping LSTM section.')
else:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

    df_base = datasets['Standard 60Ah']
    X_seq, y_seq = build_sequences(df_base, window_size=WINDOW_SIZE)
    split_s = int(0.8 * len(X_seq))
    X_tr_s, X_te_s = X_seq[:split_s], X_seq[split_s:]
    y_tr_s, y_te_s = y_seq[:split_s], y_seq[split_s:]
    print(f'Sequences — Train: {X_tr_s.shape}  |  Test: {X_te_s.shape}')

In [ ]:
if tensorflow_available():
    tf.random.set_seed(SEED)

    # ── Stacked LSTM architecture ─────────────────────────────────────────────
    def build_stacked_lstm(window_size, feature_count, units=64, dropout=0.2, lr=1e-3):
        model = Sequential([
            LSTM(units, return_sequences=True,
                 input_shape=(window_size, feature_count),
                 name='lstm_1'),
            Dropout(dropout, name='drop_1'),
            LSTM(units // 2, return_sequences=False, name='lstm_2'),
            Dropout(dropout, name='drop_2'),
            BatchNormalization(name='bn'),
            Dense(32, activation='relu', name='fc_1'),
            Dense(1, activation='sigmoid', name='output')  # sigmoid → [0,1]
        ])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(lr),
            loss='mse',
            metrics=['mae']
        )
        return model

    stacked_lstm = build_stacked_lstm(
        window_size=WINDOW_SIZE, feature_count=2, units=64, dropout=0.2)
    stacked_lstm.summary()

In [ ]:
if tensorflow_available():
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True,
                      verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    ]

    history_s = stacked_lstm.fit(
        X_tr_s, y_tr_s,
        validation_split=0.15,
        epochs=30,
        batch_size=64,
        callbacks=callbacks,
        verbose=1
    )

In [ ]:
if tensorflow_available():
    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_s.history['loss'], label='Train')
    axes[0].plot(history_s.history['val_loss'], label='Val')
    axes[0].set_title('Stacked LSTM — Loss (MSE)')
    axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(history_s.history['mae'], label='Train')
    axes[1].plot(history_s.history['val_mae'], label='Val')
    axes[1].set_title('Stacked LSTM — MAE')
    axes[1].set_xlabel('Epoch'); axes[1].legend()

    plt.tight_layout(); plt.show()

    y_stacked_pred = np.clip(stacked_lstm.predict(X_te_s, verbose=0).flatten(), 0, 1)
    print('── Stacked LSTM Results ──────────────────────────────')
    print(f'RMSE : {rmse(y_te_s, y_stacked_pred):.4f}')
    print(f'MAE  : {mae(y_te_s, y_stacked_pred):.4f}')
    print(f'MAPE : {mape(y_te_s, y_stacked_pred):.2f} %')

---
## 6. Residual & Error Analysis

In [ ]:
if tensorflow_available():
    residuals = y_te_s - y_stacked_pred

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Residuals over time
    axes[0].plot(residuals, lw=0.6, color='steelblue', alpha=0.7)
    axes[0].axhline(0, color='k', lw=0.8, ls='--')
    axes[0].set_title('Residuals over Test Steps')
    axes[0].set_xlabel('Step'); axes[0].set_ylabel('Error (True − Pred)')

    # Residual histogram
    axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='k', lw=0.8, ls='--')
    axes[1].set_title('Residual Distribution')
    axes[1].set_xlabel('Error')

    # Error vs SoC level
    axes[2].scatter(y_te_s, np.abs(residuals), s=2, alpha=0.3, color='darkorange')
    axes[2].set_title('|Error| vs True SoC')
    axes[2].set_xlabel('True SoC'); axes[2].set_ylabel('|Error|')

    plt.suptitle('Stacked LSTM — Error Analysis', fontsize=13)
    plt.tight_layout(); plt.show()

---
## 7. Coulomb Counting Drift Comparison

Coulomb counting is the classic naive method. We show how it drifts versus our ML approach.

In [ ]:
cfg_std = CONFIGS['Standard 60Ah']
df_cc = datasets['Standard 60Ah'].copy()

# Simulate Coulomb counting with 2% current noise
rng = np.random.default_rng(SEED)
current_noisy = df_cc['current_a'] + rng.normal(0, 0.02 * df_cc['current_a'].abs().mean(),
                                                  len(df_cc))
charge_consumed = (current_noisy * cfg_std.dt_s / 3600).cumsum()
soc_cc = cfg_std.soc_init - charge_consumed / cfg_std.capacity_ah
soc_cc = np.clip(soc_cc, 0, 1)

t = df_cc['time_s'].values / 60

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(t, df_cc['soc'],  'k-',  lw=1.5, label='True SoC')
ax.plot(t, soc_cc, 'r--', lw=1.0, label=f'Coulomb Counting (RMSE={rmse(df_cc["soc"].values, soc_cc):.4f})')

if tensorflow_available():
    # Re-align stacked pred for full-series plot
    pad = np.full(WINDOW_SIZE, np.nan)
    soc_lstm_full = np.concatenate([pad, stacked_lstm.predict(
        build_sequences(df_cc, window_size=WINDOW_SIZE)[0], verbose=0
    ).flatten()])
    soc_lstm_full = np.clip(soc_lstm_full, 0, 1)
    ax.plot(t, soc_lstm_full[:len(t)], 'b-', lw=1.0, alpha=0.8,
            label=f'Stacked LSTM (RMSE={rmse(df_cc["soc"].values[WINDOW_SIZE:], soc_lstm_full[WINDOW_SIZE:len(t)]):.4f})')

ax.set_xlabel('Time (min)'); ax.set_ylabel('SoC')
ax.set_title('SoC Estimation: Coulomb Counting vs Stacked LSTM')
ax.legend()
plt.tight_layout(); plt.show()

---
## 8. Summary & Next Steps

In [ ]:
print('Week 3 Updated — Summary')
print('='*55)
print('Key improvements over Week 3 baseline:')
print('  1. GridSearch CV tuning → found optimal RF hyperparams')
print('  2. Stacked LSTM (2-layer + Dropout + BN) with EarlyStopping')
print('  3. Wider window (60 s) captures more temporal context')
print('  4. Coulomb Counting drift benchmark')
print()
print('Cross-config robustness:')
print('  Standard → good; Aged battery shows higher error')
print('  → Need chemistry-aware features for aged cells')
print()
print('Next (Week 4):')
print('  • SPM (Single Particle Model) physics-based features')
print('  • Transformer architecture for long-range temporal modelling')
print('  • SoH estimation via cycle-aging simulation')